In [23]:
import allel
print("allel:",allel.__version__)
import numpy as np
print("numpy:",np.__version__)

import re
import sys

from tabulate import tabulate
import pandas as pd

import tabix

from math import isnan, isfinite

allel: 1.3.5
numpy: 1.21.6


In [2]:
from plotnine import *

In [3]:
ldfile='aegy.wgs.non-AfrI3.AfrI3.unrel.n1206.mapQ20.JntSNPs.dMM4.Ngoye.6.1MBs.beagle.gz.mm05.50kb.rnd10.ld'

metafile='resources/meta_Aaeg1kg_spp.txt'
invfile='resources/lostruct_inversions_table.txt'


In [4]:
meta = pd.read_csv(metafile, sep='\t')
meta = np.genfromtxt(metafile,delimiter='\t',names=True,dtype=None,encoding='utf-8')


In [5]:
invs = pd.read_csv(invfile, sep='\t')
invs

,call_id,inv_name,chr,start,end,size,f,f_Afr,f_Row
0,X1_Gabon_1,1a,1,1500000,17000000,15.5,0.157,0.378,0.009
1,X1_global_1,1b,1,148000000,160500000,12.5,0.117,0.237,0.037
2,X1_PuertoRico_1,1c,1,107000000,109500000,2.5,0.024,0.015,0.030
3,X1_PuertoRico_3,1d,1,248500000,252000000,3.5,0.056,0.021,0.080
4,X1_Senegal_1,1e,1,264500000,309000000,44.5,0.025,0.062,0.000
5,X1_Uganda_2,1f,1,302500000,308000000,5.5,0.060,0.123,0.018
6,X1_Wafrica_1,1g,1,162000000,170500000,8.5,0.076,0.096,0.063
7,X2_BurkinaFaso_1,1a,2,464500000,473000000,8.5,0.002,0.004,0.000
8,X2_SaudiArabia_1,1b,2,227000000,239000000,12.0,0.177,0.036,0.271
9,X2_Senegal_1,1c,2,306500000,317000000,10.5,0.566,0.205,0.808


In [6]:
is1=0
is2=6
idist = 12
ir2 = 16

chrtr = {"NC_035107.1":1,
        "NC_035108.1":2,
        "NC_035109.1":3}


In [39]:
i=0

with open(ldfile) as f:
    for l in f:
        i+=1
        c = l.split()
        c1,p1 = c[is1].split(":")
        c2,p2 = c[is2].split(":")
        c1i = chrtr[c1]
        c2i = chrtr[c2]
        p1 = int(p1)
        p2 = int(p2)
        dist = int(c[idist])
        r2 = float(c[ir2])
        if(isnan(r2)): continue
        if(c1 != c2): continue
        if i > 50: break
        print("\t".join(map(str,[c1i,p1,p2,dist,r2])))
        
        
        

In [44]:
#ld = np.genfromtxt(ldfile,delimiter='\t',dtype=None,encoding='utf-8')
ldmin='aegy.wgs.non-AfrI3.AfrI3.unrel.n1206.mapQ20.JntSNPs.dMM4.Ngoye.6.1MBs.beagle.gz.mm05.50kb.rnd10.SORT.ld.gz'

tb = tabix.open(ldmin)

bsize=500000
ldwin=50000
for chrom in map(str,[1,2,3]):
    for st in range(0,400000000,500000):    
        en = st + bsize
        # These queries are identical. A query returns an iterator over the results.
        ldtab = tb.query(chrom, st-ldwin, en+ldwin)
        allr2 = list()
        for line in ldtab:
            c1i,p1,p2,dist,r2 = line
            p1 = int(p1)
            p2 = int(p2)
            r2 = float(r2)
            if (p1>=st & p1<en) | (p2>=st & p2<en):
                if(isfinite(r2)):
                    allr2.append(r2)
        meanr2 = np.nanmean(allr2)
        if not isnan(meanr2):
            print(c1i,st,en,meanr2)


/var/folders/33/h3nj351j6fgd7_jh5_cjk60r0000gn/T/ipykernel_85535/3613190720.py:22: RuntimeWarning: Mean of empty slice


1 99500000 100000000 0.0949010024273892
1 100000000 100500000 0.08490052380034392
1 100500000 101000000 0.08150935404690762
1 101000000 101500000 0.1038396888070575
1 249500000 250000000 0.10659810399346865
1 250000000 250500000 0.12563460770872947
1 250500000 251000000 0.11036800612875806
1 251000000 251500000 0.10910582575689258
2 99500000 100000000 0.08769702487107076
2 100000000 100500000 0.08706478491301904
2 100500000 101000000 0.08366216182314537
2 101000000 101500000 0.08725287021083475
2 299500000 300000000 0.09570702681678606
2 300000000 300500000 0.11316603741180728
2 300500000 301000000 0.12143665199428899
2 301000000 301500000 0.11537227599806141
3 99500000 100000000 0.10289787721725559
3 100000000 100500000 0.09390113695421005
3 100500000 101000000 0.0900059637369884
3 101000000 101500000 0.08581111405345164
3 299500000 300000000 0.11131736148049615
3 300000000 300500000 0.11116169820012359
3 300500000 301000000 0.1158897625001457
3 301000000 301500000 0.13723372289800734